In [ ]:
import pandas as pd
import altair as alt
from ecostyles import EcoStyles

styles = EcoStyles(); styles.register_and_enable_theme()

df = pd.DataFrame({
    "age": ["16-24","25-34","35-44","45-54","55-64","65-74","75+"],
    "share": [0, 0.01, 0.48, 1.46, 21.51, 34.23, 42.31],
})
df["hi"] = df["age"].isin(["65-74","75+"])

bars = alt.Chart(df).mark_bar(cornerRadius=2).encode(
    x=alt.X("age:N", sort=df["age"].tolist(), title="Age of household reference person",
            axis=alt.Axis(labelAngle=0)),
    y=alt.Y("share:Q", title="Share of directly-held UK bonds and gilts (%)",
            scale=alt.Scale(domain=[0, 50])),
    color=alt.Color("hi:N", scale=alt.Scale(domain=[True, False], range=["#E6224B", "#36B7B4"]),
                    legend=None))
labels = alt.Chart(df[df.share >= 1]).mark_text(dy=-6, fontSize=11, color="#122b39").encode(
    x=alt.X("age:N", sort=df["age"].tolist()), y="share:Q", text=alt.Text("share:Q", format=".0f"))

caption = alt.Title(
    subtitle=[
        "Share of all directly-held UK bonds and gilts, by age of household reference person, %.",
        "Over three-quarters are held by households headed by someone aged 65 or over.",
    ],
    orient="bottom", anchor="start", fontSize=11, subtitleFontSize=10,
    color="#676A86", subtitleColor="#676A86", dy=12)

chart = (bars + labels).properties(width=640, height=340, title=caption).configure(
    background="white", font="Circular Std").configure_view(fill="transparent", stroke="transparent")
styles.save(chart, path=".", name="gilt_by_age", svg=True)
chart.save("gilt_by_age.png", scale_factor=2.0)
chart

alt.LayerChart(...)

In [ ]:
import pandas as pd
import altair as alt
from ecostyles import EcoStyles

styles = EcoStyles(); styles.register_and_enable_theme()

GAP = " "  # blank category that creates the visual break between the two groups

decile = pd.DataFrame({
    "cat": [str(d) for d in range(1, 11)],
    "share": [0, 0.03, 0, 0.34, 0.27, 1.64, 8.36, 4.51, 21.12, 63.72],
})
decile["hi"] = decile["cat"] == "10"

age = pd.DataFrame({
    "cat": ["16-24", "25-34", "35-44", "45-54", "55-64", "65-74", "75+"],
    "share": [0, 0.01, 0.48, 1.46, 21.51, 34.23, 42.31],
})
age["hi"] = age["cat"].isin(["65-74", "75+"])

df = pd.concat([decile, age], ignore_index=True)
order = decile["cat"].tolist() + [GAP] + age["cat"].tolist()

x = alt.X("cat:O", sort=order, scale=alt.Scale(domain=order), title=None,
          axis=alt.Axis(labelAngle=0, ticks=False))
y_scale = alt.Scale(domain=[0, 70])
y = alt.Y("share:Q", scale=y_scale,
          title="Share of directly-held UK bonds and gilts (%)")

# faint dashed divider through the gap
divider = alt.Chart(pd.DataFrame({"cat": [GAP]})).mark_rule(
    color="#cfd4d9", strokeDash=[3, 3]).encode(x=x)

bars = alt.Chart(df).mark_bar(cornerRadius=2).encode(
    x=x, y=y,
    color=alt.Color("hi:N",
                    scale=alt.Scale(domain=[True, False], range=["#E6224B", "#36B7B4"]),
                    legend=None))

labels = alt.Chart(df[df["share"] >= 1]).mark_text(
    dy=-6, fontSize=11, color="#122b39").encode(
    x=x, y=alt.Y("share:Q", scale=y_scale), text=alt.Text("share:Q", format=".0f"))

# in-plot group titles (these replace the two x-axis titles)
def note(cat, ypos, text, weight="normal", color="#122b39"):
    return alt.Chart(pd.DataFrame({"cat": [cat], "share": [ypos], "t": [text]})).mark_text(
        align="left", dx=-14, fontSize=11, fontWeight=weight, color=color).encode(
        x=x, y=alt.Y("share:Q", scale=y_scale), text="t:N")

notes = (
    note("1", 68, "By household wealth decile", weight="bold")
    + note("1", 63.5, "1 = least wealthy, 10 = most", color="#676A86")
    + note("16-24", 68, "By age of household reference person", weight="bold")
)

caption = alt.Title(
    subtitle=[
        "Share of all directly-held UK bonds and gilts, by household wealth decile and by age of household reference person, %.",
        "The wealthiest tenth hold nearly two-thirds; households aged 65+ hold over three-quarters.",
    ],
    orient="bottom", anchor="start", fontSize=11, subtitleFontSize=10,
    color="#676A86", subtitleColor="#676A86", dy=12)

chart = (divider + bars + labels + notes).properties(
    width=680, height=340, title=caption).configure(
    background="white", font="Circular Std").configure_view(
    fill="transparent", stroke="transparent")

styles.save(chart, path=".", name="gilt_by_wealth_decile_and_age", svg=True)
chart.save("gilt_by_wealth_decile_and_age.png", scale_factor=2.0)
chart

alt.LayerChart(...)

In [21]:
import numpy as np
import pandas as pd
import altair as alt
from ecostyles import EcoStyles

styles = EcoStyles(); styles.register_and_enable_theme()

COL, INK, GREY = "#36B7B4", "#122b39", "#676A86"   # one colour throughout

decile = pd.DataFrame({
    "label": [str(d) for d in range(1, 11)],
    "share": [0, 0.03, 0, 0.34, 0.27, 1.64, 8.36, 4.51, 21.12, 63.72]})
age = pd.DataFrame({
    "label": ["16-24", "25-34", "35-44", "45-54", "55-64", "65-74", "75+"],
    "share": [0, 0.01, 0.48, 1.46, 21.51, 34.23, 42.31]})



def caption(*lines, source=None):
    """Returns kwargs for .properties(): a title, or nothing at all."""
    lines = [l for l in lines if l]
    style = dict(orient="bottom", anchor="start", fontSize=11, subtitleFontSize=10,
                 color=GREY, subtitleColor=GREY, dy=12)
    if source:
        return {"title": alt.Title(text=source, subtitle=lines or alt.Undefined, **style)}
    if lines:   # no source: first line becomes the title, the rest the subtitle
        return {"title": alt.Title(text=lines[0], subtitle=lines[1:] or alt.Undefined, **style)}
    return {}


def finish(chart, name):
    chart = chart.configure(background="white", font="Circular Std").configure_view(
        fill="transparent", stroke="transparent")
    styles.save(chart, path=".", name=name, svg=True)
    chart.save(f"{name}.png", scale_factor=2.0)
    return chart


# =====================================================================
# 1. "100 gilts" waffle: each square = 1% of all directly-held gilts
# =====================================================================
CELL, GAP_X = 28, 12          # px per grid unit; second grid starts 12 units along

wealth_tiers = [("Wealthiest 10% of households", decile.share.iloc[9]),
                ("Next wealthiest 10%",           decile.share.iloc[8]),
                ("Poorest 80%",                   decile.share.iloc[:8].sum())]
age_tiers = [("Aged 75+",    age.share.iloc[6]),
             ("Aged 65-74",  age.share.iloc[5]),
             ("Under 65",    age.share.iloc[:5].sum())]
OPS = [1, 0.45, 0.15]         # same hue, three tints

def waffle(tiers, x0):
    vals = np.array([v for _, v in tiers])
    counts = np.diff(np.round(vals.cumsum() / vals.sum() * 100).astype(int), prepend=0)
    i = np.arange(100)
    d = pd.DataFrame({"x": i % 10 + x0 + 0.07, "y": i // 10 + 0.07, "op": np.repeat(OPS, counts)})
    d["x2"], d["y2"] = d.x + 0.86, d.y + 0.86
    return d

def tier_key(tiers, x0):
    return pd.DataFrame({
        "x": x0 + 0.07, "x2": x0 + 0.93,
        "ky": [11.0, 12.0, 13.0], "y": [10.6, 11.6, 12.6], "y2": [11.4, 12.4, 13.4],
        "pct": [f"{v:.0f}%" for _, v in tiers], "name": [n for n, _ in tiers],
        "op": OPS, "px": x0 + 1.2, "nx": x0 + 3.1})

cells = pd.concat([waffle(wealth_tiers, 0), waffle(age_tiers, GAP_X)])
key = pd.concat([tier_key(wealth_tiers, 0), tier_key(age_tiers, GAP_X)])
heads = pd.DataFrame({"x": [0.07, GAP_X + 0.07], "y": -1.0,
                      "t": ["By household wealth", "By age of household reference person"]})

X = alt.X("x:Q", scale=alt.Scale(domain=[0, 22]), axis=None)
Y = alt.Y("y:Q", scale=alt.Scale(domain=[-2, 14], reverse=True), axis=None)
op = alt.Opacity("op:Q", scale=None, legend=None)

squares = alt.Chart(cells).mark_rect(cornerRadius=3, color=COL).encode(
    x=X, x2="x2:Q", y=Y, y2="y2:Q", opacity=op)
swatches = alt.Chart(key).mark_rect(cornerRadius=3, color=COL).encode(
    x=X, x2="x2:Q", y=Y, y2="y2:Q", opacity=op)

def text(df, xcol, field, **kw):
    return alt.Chart(df).mark_text(align="left", baseline="middle", color=INK, **kw).encode(
        x=alt.X(xcol, type="quantitative", scale=alt.Scale(domain=[0, 22]), axis=None),
        y=alt.Y("ky:Q", scale=alt.Scale(domain=[-2, 14], reverse=True), axis=None),
        text=f"{field}:N")

head_txt = alt.Chart(heads).mark_text(
    align="left", baseline="middle", fontSize=12, fontWeight="bold", color=INK).encode(
    x=X, y=Y, text="t:N")

waffle_chart = alt.layer(
    squares, swatches,
    text(key, "px", "pct", fontSize=14, fontWeight="bold"),
    text(key, "nx", "name", fontSize=11),
    head_txt,
).properties(
    width=22 * CELL, height=16 * CELL,
    **caption("Each square is 1% of all directly-held UK bonds and gilts."))

finish(waffle_chart, "gilt_waffle")


# =====================================================================
# 2. Same 100%, two cuts: stacked bars on one shared axis
# =====================================================================
def segments(df, row_y, prefix):
    d = df.copy()
    d["x2"] = d.share.cumsum(); d["x"] = d.x2 - d.share; d["mid"] = (d.x + d.x2) / 2
    d["op"] = 0.12 + 0.88 * np.arange(len(d)) / (len(d) - 1)   # light -> dark
    d["y"], d["y2"] = row_y, row_y + 1
    d["ymid"] = row_y + 0.5
    d["pct"] = d.share.map(lambda s: f"{s:.0f}%")
    d["name"] = prefix + d.label
    return d

seg = pd.concat([segments(decile, 0.0, "Decile "), segments(age, 2.5, "Aged ")])
big = seg[seg.share >= 15]

# brackets for the "long tail" of tiny slices
br = pd.DataFrame({
    "x": 0, "x2": [decile.share.iloc[:8].sum(), age.share.iloc[:5].sum()],
    "y": [1.2, 3.6], "ty": [1.5, 3.9],
    "t": [f"Poorest 80% of households: {decile.share.iloc[:8].sum():.0f}% between them",
          f"Aged under 65: {age.share.iloc[:5].sum():.0f}% between them"]})
rows = pd.DataFrame({"x": 0, "y": [-0.35, 2.1],
                     "t": ["By household wealth decile",
                           "By age of household reference person"]})

XSC = alt.Scale(domain=[0, 100], nice=False)
XAX = alt.Axis(title="Share of all directly-held UK bonds and gilts (%)",
               values=[0, 25, 50, 75, 100], grid=False)
YSC = alt.Scale(domain=[-0.8, 4.3], reverse=True)
X2 = alt.X("x:Q", scale=XSC, axis=XAX)
Y2 = alt.Y("y:Q", scale=YSC, axis=None)

bars = alt.Chart(seg).mark_rect(color=COL, stroke="white", strokeWidth=1).encode(
    x=X2, x2="x2:Q", y=Y2, y2="y2:Q", opacity=alt.Opacity("op:Q", scale=None, legend=None))

def lab(df, field, **kw):
    return alt.Chart(df).mark_text(color=INK, **kw).encode(
        x=alt.X("mid:Q", scale=XSC, axis=XAX), y=alt.Y("ymid:Q", scale=YSC, axis=None), text=f"{field}:N")

pct_txt  = lab(big, "pct", fontSize=20, fontWeight="bold", dy=-8)
name_txt = lab(big, "name", fontSize=11, dy=12)

rule = alt.Chart(br).mark_rule(color=GREY, strokeWidth=1.5).encode(x=X2, x2="x2:Q", y=Y2)
br_txt = alt.Chart(br).mark_text(align="left", baseline="middle", fontSize=11, color=GREY).encode(
    x=X2, y=alt.Y("ty:Q", scale=YSC, axis=None), text="t:N")
row_txt = alt.Chart(rows).mark_text(align="left", baseline="middle", fontSize=12,
                                    fontWeight="bold", color=INK).encode(x=X2, y=Y2, text="t:N")

stack_chart = alt.layer(bars, pct_txt, name_txt, rule, br_txt, row_txt).properties(
    width=640, height=306,
    **caption())

finish(stack_chart, "gilt_two_cuts")


# =====================================================================
# 3. Ranked dot plot: all 17 slices, filled = wealth decile, hollow = age
# =====================================================================
rank = pd.concat([
    decile.assign(name="Wealth decile " + decile.label, kind="Wealth decile"),
    age.assign(name="Age " + age.label, kind="Age band")], ignore_index=True)
rank["lab"] = rank.share.map(lambda s: f"{s:.0f}%" if s >= 1 else "<1%")
order = rank.sort_values("share", ascending=False, kind="stable").name.tolist()

XS = alt.Scale(domain=[0, 70])
XR = alt.X("share:Q", scale=XS, axis=alt.Axis(title="Share of directly-held UK bonds and gilts (%)"))
YR = alt.Y("name:N", sort=order, axis=alt.Axis(title=None, ticks=False, domain=False, labelPadding=8))
fill = alt.Fill("kind:N", scale=alt.Scale(domain=["Wealth decile", "Age band"], range=[COL, "white"]),
                legend=None)

stems = alt.Chart(rank).mark_rule(color=COL, strokeWidth=2, opacity=0.5).encode(
    x=XR, x2=alt.X2Datum(0), y=YR)
dots = alt.Chart(rank).mark_circle(size=120, stroke=COL, strokeWidth=2, opacity=1).encode(
    x=XR, y=YR, fill=fill)
vals = alt.Chart(rank).mark_text(align="left", dx=10, fontSize=11, color=INK).encode(
    x=XR, y=YR, text="lab:N")

# in-plot key (replaces a legend)
k = pd.DataFrame({"name": [order[11], order[13]], "share": 36,
                  "kind": ["Wealth decile", "Age band"],
                  "t": ["By household wealth decile", "By age of household reference person"]})
key_dots = alt.Chart(k).mark_circle(size=120, stroke=COL, strokeWidth=2, opacity=1).encode(
    x=XR, y=YR, fill=fill)
key_txt = alt.Chart(k).mark_text(align="left", dx=14, fontSize=11, color=INK).encode(
    x=XR, y=YR, text="t:N")

dot_chart = alt.layer(stems, dots, vals, key_dots, key_txt).properties(
    width=600, height=17 * 22,
    **caption())

finish(dot_chart, "gilt_ranked")

alt.LayerChart(...)